# Predictive checklists: Gleason + TP53/RB1/PTEN + labs

5-year platinum and NEPC checklists on the `gleason_somatic_available_case` subset: patients with a sequencing report and a Gleason score, built by `build_somatic_gleason_inputs.py` (notebook 01, Stage 3b). This subset runs on the **sequencing clock**: time 0 is the report date (`REPORT_DT`) of the specimen closest to ADT start, landmarks are sequencing + 0/90/180 days, labs and Gleason must be available by then, outcomes run from there, and the base split is inherited. Age is age at sequencing.

Candidates are the standard lab summaries and age, plus `GLEASON_SCORE` and `TP53_ANY` / `RB1_ANY` / `PTEN_ANY` (any SNV/DEL/AMP/SV on the anchoring specimen), each thresholded at its observed training values. The matched Cox/XGBoost comparators get the same predictors. Settings are in `gleason_trio_config.json`; `min_events` is 5 (not 10) because NEPC events in this subset are few, and the resolved-label NEPC sensitivity is off. See `README.md` for the IPCW estimand and leakage boundaries; this notebook only drives the CLI.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "survival_common").is_dir())
SUBPROJECT = REPO_ROOT / "COMPASS/survival_analysis/predictive_checklists"
INPUTS_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
SUBSET = "somatic_gleason/gleason_somatic_available_case"
PLATINUM_INPUTS = INPUTS_ROOT / "prediction_inputs_adt" / SUBSET
NEPC_INPUTS = INPUTS_ROOT / "prediction_inputs_adt_nepc" / SUBSET
OUTPUT_ROOT = INPUTS_ROOT / "predictive_checklists"
CONFIG = SUBPROJECT / "gleason_trio_config.json"
LANDMARKS = [0, 90, 180]  # sequencing + D days
HORIZONS = [1825]  # 5 years
RUN_PREFLIGHT = True
RUN_TRAINING = False
OVERWRITE = False  # archives previous generated results
RESUME = False  # training only: reuse completed horizons of an interrupted run


## Validate cohort inputs

Checks both endpoint subsets at every landmark and reports the 5-year known cases/controls per split. Review the NEPC counts before training: an underpowered horizon is recorded as such, not fitted.

In [ ]:
base = [sys.executable, "-m", "COMPASS.survival_analysis.predictive_checklists",
        "--config", str(CONFIG),
        "--platinum-inputs", str(PLATINUM_INPUTS), "--nepc-inputs", str(NEPC_INPUTS),
        "--landmarks", *map(str, LANDMARKS), "--horizons", *map(str, HORIZONS)]
extra = ["--overwrite"] if OVERWRITE else []
if RUN_PREFLIGHT:
    subprocess.run(base + ["--output-dir", str(OUTPUT_ROOT / "preflight_gleason_trio"), "--preflight-only"] + extra,
                   cwd=REPO_ROOT, check=True)


In [ ]:
import json
import pandas as pd

preflight_path = OUTPUT_ROOT / "preflight_gleason_trio" / "preflight.json"
if preflight_path.exists():
    rows = []
    for cell in json.loads(preflight_path.read_text()):
        for horizon in cell.get("requested_horizons", [{}]):
            rows.append({"endpoint": cell["endpoint"], "landmark_days": cell["landmark_days"],
                         "n_patients": cell.get("n_patients"), "events_by_split": cell.get("events_by_split"),
                         "horizon_days": horizon.get("horizon_days"),
                         "status": horizon.get("status", cell["status"]),
                         "reason": horizon.get("reason", cell.get("reason"))})
    display(pd.DataFrame(rows))


## Train and evaluate

Set `RUN_TRAINING=True` after reviewing preflight. The test block remains held out.

In [ ]:
RUN_OUTPUT = OUTPUT_ROOT / "gleason_trio_labs"
if RUN_TRAINING:
    resume = ["--resume"] if RESUME and not OVERWRITE else []
    subprocess.run(base + ["--output-dir", str(RUN_OUTPUT)] + extra + resume,
                   cwd=REPO_ROOT, check=True)


## Inspect results and render figures

One 5-year classifier per endpoint × landmark; the selected items show whether Gleason or the trio entered alongside labs. These are horizon-specific results on the sequencing clock and are not comparable patient-for-patient with the full-ADT checklist run.

In [ ]:
metrics_path = RUN_OUTPUT / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics[[c for c in ["endpoint", "landmark_days", "horizon_days", "model", "status",
                                 "auc", "brier", "n_items", "reason"] if c in metrics]])
    for checklist in sorted(RUN_OUTPUT.glob("*/landmark_*/full/horizon_*/checklist.txt")):
        print(f"--- {checklist.relative_to(RUN_OUTPUT)}")
        print(checklist.read_text())
    subprocess.run(["Rscript", str(SUBPROJECT / "render_figures.R"), str(RUN_OUTPUT)],
                   cwd=REPO_ROOT, check=True)
